In [ ]:
import pandas as pd
import numpy as np 
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split,KFold
from sklearn.feature_selection import RFE
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,auc
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

## Data Setup

Download the dataset from Kaggle: [Telecom Churn Case Study Hackathon](https://www.kaggle.com/competitions/telecom-churn-case-study-hackathon-c55/data)

Place `train.csv` and `test.csv` inside the `data/` folder before running this notebook.

```
telecom-churn-prediction/
├── data/
│   ├── train.csv
│   └── test.csv
└── notebooks/
    └── telecom_churn_case_study.ipynb
```

In [ ]:
# Data should be placed in the data/ folder
# Download from: https://www.kaggle.com/competitions/telecom-churn-case-study-hackathon-c55/data
path = 'data/train.csv'
df = pd.read_csv(path)


In [ ]:
df.head()

In [ ]:
df

In [ ]:
df['churn_probability'].value_counts()

In [ ]:
df.info(verbose=1)

In [ ]:
df.isnull().sum().to_dict()

In [ ]:
(df.isnull().sum()/len(df)).to_dict()

In [ ]:
# there is only a single circle id in the dataset , reomve the column
df['circle_id'].unique()


In [ ]:
df = df.drop(columns = ['circle_id'])

In [ ]:
df.describe()

In [ ]:
# og_t2o_mou cols 
drop_og_t2o_mou = df.columns[df.columns.str.contains(r't2o_mou$')]
df = df.drop(columns = drop_og_t2o_mou)

In [ ]:
df.shape

### ARPU - Average Revenue Per User

In [ ]:
arpu_cols = df.columns[df.columns.str.contains(r'arpu')]
arpu_cols

In [ ]:
# Create total average revenue per user column , do this monthly
df['total_arpu_6'] = df['arpu_6'] + df['arpu_3g_6'].fillna(0) + df['arpu_2g_6'].fillna(0)
df['total_arpu_7'] = df['arpu_7'] + df['arpu_3g_7'].fillna(0) + df['arpu_2g_7'].fillna(0)
df['total_arpu_8'] = df['arpu_8'] + df['arpu_3g_8'].fillna(0) + df['arpu_2g_8'].fillna(0)

In [ ]:
total_arpu_cols = df.columns[df.columns.str.contains(r'^total_arpu_')]
total_arpu_cols

In [ ]:
df['total_arpu_monthly'] = df[total_arpu_cols].sum(axis=1) / len(total_arpu_cols)

In [ ]:
total_arpu_cols = df.columns[df.columns.str.contains(r'(^total_arpu_)')]
total_arpu_cols

- It can be observed that the arpu is lower for customers who are likely to churn and customers who don't churn seem to have a higher arpu
- Another thing to be noticed as a general trend is that customers who don't churn increase or maintain their revenue over the months , while customers that churn generate lesser and lesser revenue


In [ ]:
df.loc[df.churn_probability == 1,list(total_arpu_cols) + ['churn_probability']].describe()

In [ ]:
df.loc[df.churn_probability == 0,list(total_arpu_cols) + ['churn_probability']].describe()

In [ ]:
df['arpu_diff_7_6'] = df['total_arpu_7'] - df['total_arpu_6']
df['arpu_diff_8_7'] = df['total_arpu_8'] - df['total_arpu_7']
df['arpu_diff_8_6'] = df['total_arpu_8'] - df['total_arpu_6']

- It can be observed that customers who 

In [ ]:
# fig,ax = plt.subplots()
# ax2 = ax.twinx()
# plt.plot(df['arpu_diff_7_6'],kind='dist')
df2 = df.loc[(df['arpu_diff_7_6'] > df['arpu_diff_7_6'].quantile(.05)) & (df['arpu_diff_7_6'] < df['arpu_diff_7_6'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='arpu_diff_7_6',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='arpu_diff_7_6',color='b')

In [ ]:
df2 = df.loc[(df['arpu_diff_8_7'] > df['arpu_diff_8_7'].quantile(.05)) & (df['arpu_diff_8_7'] < df['arpu_diff_8_7'].quantile(.95))]
# sns.kdeplot(data = df2[df2.churn_probability == 1],x='arpu_diff_7_6',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 1],x='arpu_diff_8_7',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='arpu_diff_8_7',color='b')

In [ ]:
df2 = df.loc[(df['arpu_diff_7_6'] > df['arpu_diff_7_6'].quantile(.05)) & (df['arpu_diff_7_6'] < df['arpu_diff_7_6'].quantile(.95))]
sns.boxplot(data=df2,y='arpu_diff_7_6',x='churn_probability')

In [ ]:
df2 = df.loc[(df['arpu_diff_8_7'] > df['arpu_diff_8_7'].quantile(.05)) & (df['arpu_diff_8_7'] < df['arpu_diff_8_7'].quantile(.95))]
sns.boxplot(data=df2,y='arpu_diff_8_7',x='churn_probability')

In [ ]:
# fig,ax = plt.subplots()
# ax2 = ax.twinx()
# plt.plot(df['arpu_diff_8_6'],kind='dist')
df2 = df.loc[(df['arpu_diff_8_6'] > df['arpu_diff_8_6'].quantile(.05)) & (df['arpu_diff_8_6'] < df['arpu_diff_8_6'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='arpu_diff_8_6',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='arpu_diff_8_6',color='b')

In [ ]:
df2 = df.loc[(df['arpu_diff_8_6'] > df['arpu_diff_8_6'].quantile(.05)) & (df['arpu_diff_8_6'] < df['arpu_diff_8_6'].quantile(.95))]
sns.boxplot(data=df2,y='arpu_diff_8_6',x='churn_probability')

In [ ]:
df[['arpu_diff_8_6','arpu_diff_8_7','arpu_diff_7_6']].corr()

In [ ]:
df['total_revenue_per_usr'] = df[['total_arpu_6', 'total_arpu_7', 'total_arpu_8']].sum(axis=1)

- drop arpu_diff_8_6 because 8_6 and 7_6 have the least correlation

In [ ]:
df['arpu_avg_7_6'] = (df['total_arpu_7'] + df['total_arpu_6']) /2 


In [ ]:
df['arpu_phase_drift'] = df['total_arpu_8'] - df['arpu_avg_7_6']

In [ ]:
df2 = df.loc[(df['arpu_phase_drift'] > df['arpu_phase_drift'].quantile(.1)) & (df['arpu_phase_drift'] < df['arpu_phase_drift'].quantile(.9))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='arpu_phase_drift',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='arpu_phase_drift',color='b')

In [ ]:
sns.boxplot(data=df2,y='arpu_phase_drift',x='churn_probability')

In [ ]:
df = df.drop(['total_arpu_6', 'total_arpu_7', 'total_arpu_8','arpu_diff_8_6'],axis=1)

In [ ]:
df = df.drop(['arpu_6', 'arpu_7', 'arpu_8', 'arpu_3g_6', 'arpu_3g_7', 'arpu_3g_8',
       'arpu_2g_6', 'arpu_2g_7', 'arpu_2g_8'],axis=1)

### MOU - Minutes of Usage Voice Mails

In [ ]:
mou_cols = df.columns[df.columns.str.contains(r'mou')]
mou_cols

- Some of the columns here are MAR 
- So apply Missing Indicator Imputation

In [ ]:
df['missing_indicator_mins_per_usg_jun'] = np.where(df['onnet_mou_6'].isnull(),1,0)
df['missing_indicator_mins_per_usg_jul'] = np.where(df['onnet_mou_7'].isnull(),1,0)
df['missing_indicator_mins_per_usg_aug'] = np.where(df['onnet_mou_8'].isnull(),1,0)

In [ ]:
# Now that there is a missing value indicator fill null values with zero
df[mou_cols] = df[mou_cols].fillna(0)

In [ ]:
onnet_mou_cols = df.columns[df.columns.str.contains(r"^onnet_mou")]
offnet_mou_cols = df.columns[df.columns.str.contains(r"^offnet_mou")]

In [ ]:
onnet_mou_cols

In [ ]:
offnet_mou_cols

In [ ]:
df['offnet_mou_diff_7_6']= df['offnet_mou_7'] - df['offnet_mou_6']
df['offnet_mou_diff_8_7'] = df['offnet_mou_8'] - df['offnet_mou_7']

In [ ]:
df['onnet_mou_diff_7_6']= df['onnet_mou_7'] - df['onnet_mou_6']
df['onnet_mou_diff_8_7'] = df['onnet_mou_8'] - df['onnet_mou_7']

In [ ]:
df['offnet_mou_phase_drift'] = df['offnet_mou_8'] - ((df['offnet_mou_7'] + df['offnet_mou_6']) /2)

In [ ]:
df['onnet_mou_phase_drift'] = df['onnet_mou_8'] - ((df['onnet_mou_7'] + df['onnet_mou_6']) /2)

In [ ]:

df2 = df.loc[(df['offnet_mou_diff_7_6'] > df['offnet_mou_diff_7_6'].quantile(.05)) & (df['offnet_mou_diff_7_6'] < df['offnet_mou_diff_7_6'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='offnet_mou_diff_7_6',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='offnet_mou_diff_7_6',color='b')

In [ ]:

df2 = df.loc[(df['onnet_mou_diff_7_6'] > df['onnet_mou_diff_7_6'].quantile(.05)) & (df['onnet_mou_diff_7_6'] < df['onnet_mou_diff_7_6'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='onnet_mou_diff_7_6',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='onnet_mou_diff_7_6',color='b')

In [ ]:

df2 = df.loc[(df['onnet_mou_diff_8_7'] > df['onnet_mou_diff_8_7'].quantile(.05)) & (df['onnet_mou_diff_8_7'] < df['onnet_mou_diff_8_7'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='onnet_mou_diff_8_7',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='onnet_mou_diff_8_7',color='b')

In [ ]:

df2 = df.loc[(df['offnet_mou_diff_8_7'] > df['offnet_mou_diff_8_7'].quantile(.05)) & (df['offnet_mou_diff_8_7'] < df['offnet_mou_diff_8_7'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='offnet_mou_diff_8_7',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='offnet_mou_diff_8_7',color='b')

In [ ]:

df2 = df.loc[(df['offnet_mou_phase_drift'] > df['offnet_mou_phase_drift'].quantile(.1)) & (df['offnet_mou_phase_drift'] < df['offnet_mou_phase_drift'].quantile(.9))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='offnet_mou_phase_drift',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='offnet_mou_phase_drift',color='b')

In [ ]:

df2 = df.loc[(df['onnet_mou_phase_drift'] > df['onnet_mou_phase_drift'].quantile(.1)) & (df['onnet_mou_phase_drift'] < df['onnet_mou_phase_drift'].quantile(.9))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='onnet_mou_phase_drift',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='onnet_mou_phase_drift',color='b')

In [ ]:
df['onnet_mou_monthly'] = df[onnet_mou_cols].sum(axis=1) / len(onnet_mou_cols)
df['offnet_mou_monthly'] = df[offnet_mou_cols].sum(axis=1) / len(offnet_mou_cols)

In [ ]:

df2 = df.loc[(df['onnet_mou_monthly'] > df['onnet_mou_monthly'].quantile(.05)) & (df['onnet_mou_monthly'] < df['onnet_mou_monthly'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='onnet_mou_monthly',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='onnet_mou_monthly',color='b')

In [ ]:

df2 = df.loc[(df['offnet_mou_monthly'] > df['offnet_mou_monthly'].quantile(.05)) & (df['offnet_mou_monthly'] < df['offnet_mou_monthly'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='offnet_mou_monthly',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='offnet_mou_monthly',color='b')

In [ ]:
sns.scatterplot(data=df2,x='onnet_mou_monthly',y='offnet_mou_monthly',hue='churn_probability')

In [ ]:
df = df.drop(columns = list(onnet_mou_cols)+list(offnet_mou_cols))

In [ ]:
roam_mou_cols = df.columns[df.columns.str.contains(r"^roam[a-zA-Z_]+mou_[0-9]$")]
roam_mou_cols

In [ ]:
roam_mou_6 = [i for i in roam_mou_cols if re.search(r"ic|og",i) and re.search(r"6",i)]
roam_mou_7 = [i for i in roam_mou_cols if re.search(r"ic|og",i) and re.search(r"7",i)]
roam_mou_8 = [i for i in roam_mou_cols if re.search(r"ic|og",i) and re.search(r"8",i)]
df['roam_mou_6'] = df[roam_mou_6].sum(axis=1)
df['roam_mou_7'] = df[roam_mou_7].sum(axis=1)
df['roam_mou_8'] = df[roam_mou_8].sum(axis=1)

In [ ]:
roam_mou = [i for i in df.columns if re.search(r"^roam_mou_[6-8]$",i)]
df['roam_mou_monthly'] = df[roam_mou].sum(axis=1) / len(df)

In [ ]:

df2 = df.loc[(df['roam_mou_monthly'] > df['roam_mou_monthly'].quantile(.05)) & (df['roam_mou_monthly'] < df['roam_mou_monthly'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='roam_mou_monthly',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='roam_mou_monthly',color='b')

In [ ]:
q=np.arange(.75,1.,.05)

In [ ]:
df[roam_mou_cols].describe()

In [ ]:
df.groupby(['churn_probability'])[roam_mou_cols].quantile(q).T

In [ ]:
mou_6 = [i for i in roam_mou_cols if re.search(r'6$',i)]
mou_7 = [i for i in roam_mou_cols if re.search(r'7$',i)]
mou_8 = [i for i in roam_mou_cols if re.search(r'8$',i)]

In [ ]:
df[mou_6]

In [ ]:

df['roam_diff_7_6'] = df[mou_7].sum(axis=1) - df[mou_6].sum(axis=1)
df['roam_diff_8_7'] = df[mou_8].sum(axis=1) - df[mou_7].sum(axis=1)

In [ ]:
df['roam_phase_drift'] = df[mou_8].sum(axis=1) - ((df[mou_7].sum(axis=1) + df[mou_6].sum(axis=1))/2)

In [ ]:
df['roam_diff_7_6'].describe()

In [ ]:

df2 = df.loc[(df['roam_diff_7_6'] > df['roam_diff_7_6'].quantile(.5)) & (df['roam_diff_7_6'] < df['roam_diff_7_6'].quantile(.99))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='roam_diff_7_6',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='roam_diff_7_6',color='b')

In [ ]:

df2 = df.loc[(df['roam_diff_8_7'] > df['roam_diff_8_7'].quantile(.5)) & (df['roam_diff_8_7'] < df['roam_diff_8_7'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='roam_diff_8_7',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='roam_diff_8_7',color='b')

In [ ]:

df2 = df.loc[(df['roam_phase_drift'] > df['roam_phase_drift'].quantile(.5)) & (df['roam_phase_drift'] < df['roam_phase_drift'].quantile(.95))]
sns.kdeplot(data = df2[df2.churn_probability == 1],x='roam_phase_drift',color = 'r')
sns.kdeplot(data = df2[df2.churn_probability == 0],x='roam_phase_drift',color='b')

In [ ]:
[i for i in df.columns if re.search(r'roam',i)]

In [ ]:
df = df.drop(columns =[i for i in df.columns if re.search(r'roam',i)] )

In [ ]:
def transform_drop_cols(df,loc) : 
    drop_t2_cols = [i for i in df.columns if re.search(r"^{0}_(og|ic)_[a-zA-Z0-9_]+mou_[6-8]$".format(loc),i)]
    df = df.drop(drop_t2_cols,axis=1)

    loc_ig_og = [i for i in df.columns if re.search(r'{0}'.format(loc),i)]
    def extract_num(x) : 
        return x.split('_')[-1]
    mnth_lst = list(set(map(extract_num,loc_ig_og)))

    l = []
    for i in mnth_lst : 
        l.append([col for col in loc_ig_og if re.search(r"{0}_(og|ic)[a-z_]+{1}".format(loc,i),col)])
#         print(l)
    for m,c in zip(mnth_lst,l) : 
        df[f'{loc}_mou_{m}'] = df[c].sum(axis=1)
#     df = df.drop(loc_ig_og,axis=1)
    return df,loc_ig_og

In [ ]:
df2 ,loc_ig_og= transform_drop_cols(df.copy(),'loc')
df2,std_ig_og = transform_drop_cols(df2,'std')
df2,spl_ig_og = transform_drop_cols(df2,'spl')
df2,isd_ig_og = transform_drop_cols(df2,'isd')
df = df2

In [ ]:
df.groupby(['churn_probability'])[loc_ig_og].describe().T

In [ ]:
df.groupby(['churn_probability'])[std_ig_og].describe().T

In [ ]:
df.groupby(['churn_probability'])[spl_ig_og].describe().T

In [ ]:
df.groupby(['churn_probability'])[isd_ig_og].describe().T

In [ ]:
mou_cols = [loc_ig_og,
std_ig_og ,
spl_ig_og,
isd_ig_og]

In [ ]:
mou_cols

In [ ]:
diff_cols_mou_7_6 = []
diff_cols_mou_8_7 = []
phase_drift_mou = []
def mou_diff_cols(df,mou_cols) :
    prefix = mou_cols[0].split('_')[0]
    mou_6 = [i for i in mou_cols if re.search(r'6$',i)]
    mou_7 = [i for i in mou_cols if re.search(r'7$',i)]
    mou_8 = [i for i in mou_cols if re.search(r'8$',i)]
    df[f'{prefix}_mou_diff_7_6'] = df[mou_7].sum(axis=1) - df[mou_6].sum(axis=1)
    df[f'{prefix}_mou_diff_8_7'] = df[mou_8].sum(axis=1) - df[mou_7].sum(axis=1)
    df[f'{prefix}_mou_phase_drift'] = df[mou_8].sum(axis=1) - ((df[mou_7].sum(axis=1) + df[mou_6].sum(axis=1))/2)
    diff_cols_mou_7_6.append(prefix)
    diff_cols_mou_8_7.append(prefix)
    phase_drift_mou.append(prefix)
    return df
    
for cols in mou_cols : 
    df = mou_diff_cols(df,cols)

In [ ]:
diff_cols_mou_7_6 = list(map(lambda x : x + "_mou_diff_7_6" ,diff_cols_mou_7_6))

In [ ]:
diff_cols_mou_7_6

In [ ]:
diff_cols_mou_8_7

In [ ]:
def dis_plot_mou_7_6(df,i,x,y) :
    df2 = df.loc[(df[i] > df[i].quantile(x)) & (df[i] < df[i].quantile(y))]
    sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
    sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')
    plt.title(f"{i} dist plot 0s vs 1s")
    plt.legend('01')
    plt.show()

- Since the scale of the values is small only a faint trend is visible , but it is certainly peresent as observed in the describe plots

In [ ]:
dis_plot_mou_7_6(df,diff_cols_mou_7_6[0],.01,.99)

In [ ]:
dis_plot_mou_7_6(df,diff_cols_mou_7_6[1],.1,.9)

In [ ]:
dis_plot_mou_7_6(df,diff_cols_mou_7_6[2],0.1,.9)

In [ ]:
dis_plot_mou_7_6(df,diff_cols_mou_7_6[3],.75,.99)

In [ ]:
diff_cols_mou_8_7 = list(map(lambda x : x + "_mou_diff_8_7" ,diff_cols_mou_8_7))

In [ ]:
def dis_plot_mou_8_7(df,i,x,y) :
    df2 = df.loc[(df[i] > df[i].quantile(x)) & (df[i] < df[i].quantile(y))]
    sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
    sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')
    plt.title(f"{i} dist plot 0s vs 1s")
    plt.legend('01')
    plt.show()

In [ ]:
dis_plot_mou_8_7(df,diff_cols_mou_8_7[0],.01,.99)

In [ ]:
dis_plot_mou_8_7(df,diff_cols_mou_8_7[1],.1,.9)

In [ ]:
dis_plot_mou_8_7(df,diff_cols_mou_8_7[2],.1,.9)

In [ ]:
dis_plot_mou_8_7(df,diff_cols_mou_8_7[3],.75,.99)

In [ ]:
loc_cols = df.columns[df.columns.str.contains(r"^loc")]
df['loc_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

In [ ]:
loc_cols = df.columns[df.columns.str.contains(r"^std")]
df['std_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

In [ ]:
loc_cols = df.columns[df.columns.str.contains(r"^isd")]
df['isd_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

In [ ]:
loc_cols = df.columns[df.columns.str.contains(r"^spl")]
df['spl_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

In [ ]:
mou_monthly_list = df.columns[df.columns.str.contains('_mou_monthly')]
cols = 2
rows = np.ceil(len(mou_monthly_list)/cols).astype('int')
fig,ax=plt.subplots(rows,cols,figsize=(15,10))
for idx,i in enumerate(mou_monthly_list) : 
    idx1 = idx // cols
    idx2 = idx % cols
    df2 = df.loc[(df[i] > df[i].quantile(.05)) & (df[i] < df[i].quantile(.95))]
    ax2 = ax[idx1,idx2].twinx()
    sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,ax=ax[idx1,idx2],color='b')
    sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,ax=ax2,color='r')
    ax[idx1,idx2].set_title(f'{i}')
[fig.delaxes(i) for i in ax.flat if not i.has_data()]
plt.tight_layout()
plt.show()

In [ ]:
phase_drift_mou = list(map(lambda x : x + "_mou_phase_drift" ,phase_drift_mou))

In [ ]:
phase_drift_mou

In [ ]:
def dis_plot_mou_phase_drift(df,i,x,y) :
    df2 = df.loc[(df[i] > df[i].quantile(x)) & (df[i] < df[i].quantile(y))]
    sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
    sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')
    plt.title(f"{i} dist plot 0s vs 1s")
    plt.legend('01')
    plt.show()

In [ ]:
dis_plot_mou_phase_drift(df,phase_drift_mou[0],.1,.9)

In [ ]:
dis_plot_mou_phase_drift(df,phase_drift_mou[1],.1,.9)

In [ ]:
dis_plot_mou_phase_drift(df,phase_drift_mou[2],.1,.9)

In [ ]:
dis_plot_mou_phase_drift(df,phase_drift_mou[3],.75,.99)

In [ ]:
mou_cols = [i for i in df.columns if re.search(r'mou',i)]

In [ ]:
drop_mou = [i for i in  mou_cols if not re.search(r'(mou_monthly$|diff|phase_drift$)',i)]

In [ ]:
drop_mou

In [ ]:
df['last_date_of_month_6'].unique()

In [ ]:
drop_date = [i for i in df.columns if re.search(r'^last_date_of_month',i)]
df = df.drop(drop_date,axis=1)

In [ ]:
total_ic_og  = df.columns[df.columns.str.contains(r'total_(ic|og)_mou')]

In [ ]:
total_ig = [i for i in total_ic_og if re.search(r'ic',i)]
total_og = [i for i in total_ic_og if re.search(r'og',i)]

In [ ]:
drop_mou = np.setdiff1d(drop_mou,total_ic_og)

In [ ]:
total_ic_og

In [ ]:
mou_6 = df.columns[df.columns.str.contains(r'^total_(og|ic)_mou_6$')]
df['total_mou_6'] = df[mou_6].sum(axis=1)
mou_7 = df.columns[df.columns.str.contains(r'^total_(og|ic)_mou_7$')]
df['total_mou_7'] = df[mou_7].sum(axis=1)
mou_8 = df.columns[df.columns.str.contains(r'^total_(og|ic)_mou_8$')]
df['total_mou_8'] = df[mou_8].sum(axis=1)

In [ ]:
total_mou = df.columns[df.columns.str.contains(r"total_mou_[6-8]$")]
total_mou

In [ ]:
df['total_incoming_mins_usg_per_usr'] = df[total_ig].sum(axis=1)
df['total_outgoing_mins_usg_per_usr'] = df[total_og].sum(axis=1)
df['total_mins_usg_per_usr'] = df[total_mou].sum(axis=1)

In [ ]:
df = df.drop(total_ic_og,axis=1)

In [ ]:
df.groupby(['churn_probability'])[total_mou].describe().T

In [ ]:
df = df.drop(drop_mou,axis=1)

In [ ]:
df['total_mou_diff_7_6'] = df['total_mou_7'] - df['total_mou_6']
df['total_mou_diff_8_7'] = df['total_mou_8'] - df['total_mou_7']

In [ ]:
df['total_mou_phase_drift'] = df['total_mou_8'] - ((df['total_mou_7'] + df['total_mou_6'])/2)

In [ ]:
df['total_mou_monthly'] = df[total_mou].sum(axis=1)

In [ ]:
df = df.drop(total_mou,axis=1)

In [ ]:
mou_cols = [i for i in df.columns if re.search(r'mou',i)]
mou_cols

In [ ]:
df.groupby(['churn_probability'])['total_mou_diff_7_6'].describe()

In [ ]:
df.groupby(['churn_probability'])['total_mou_diff_8_7'].describe()

In [ ]:
df.groupby(['churn_probability'])['total_mou_phase_drift'].describe()

In [ ]:
i = 'total_mou_diff_7_6'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'total_mou_diff_8_7'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'total_mou_phase_drift'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
plt.figure(figsize=(15,15))
corr_cols = [i for i in mou_cols if re.search(r'(diff|phase_drift)',i)] + ['churn_probability']
corr_mat = df[corr_cols]
corr_matrix = corr_mat.corr().where(np.triu(np.ones(corr_mat.corr().shape),k=1).astype('bool'))
sns.heatmap(corr_matrix,annot=True)
plt.show()

In [ ]:
# df[['arpu_diff_8_7','total_mou_diff_7_6']].cov()

# RECH - Recharge

In [ ]:
recharge_cols = df.columns[df.columns.str.contains(r"rech")]
recharge_cols

In [ ]:
rech_num_amt = [i for i in recharge_cols if re.search(r"total_rech_(num|amt)",i)]
rech_num_amt

In [ ]:
df['amt_per_recharge_6'] = df['total_rech_amt_6'] / df['total_rech_num_6']
df['amt_per_recharge_7'] = df['total_rech_amt_7'] / df['total_rech_num_7']
df['amt_per_recharge_8'] = df['total_rech_amt_8'] / df['total_rech_num_8']

In [ ]:
amt_per_recharge_mnth = [i for i in df.columns if re.search(r'amt_per_recharge_[6-8]$',i)]

In [ ]:
df.loc[df['amt_per_recharge_6'].isnull(),['total_rech_amt_6','total_rech_num_6']]

In [ ]:
df[amt_per_recharge_mnth] = df[amt_per_recharge_mnth].fillna(0)

In [ ]:
df.groupby(['churn_probability'])[amt_per_recharge_mnth].describe().T

In [ ]:
df['amt_per_recharge_diff_7_6'] = df['amt_per_recharge_7'] - df['amt_per_recharge_6']
df['amt_per_recharge_diff_8_7'] = df['amt_per_recharge_8'] - df['amt_per_recharge_7']

In [ ]:
df['amt_per_recharge_phase_drift'] = df['amt_per_recharge_8'] - ((df['amt_per_recharge_7'] + df['amt_per_recharge_6'])/2)

In [ ]:
i = 'amt_per_recharge_diff_7_6'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'amt_per_recharge_diff_8_7'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'amt_per_recharge_phase_drift'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
df['total_rech_num_monthly'] = df.loc[:,rech_num_amt[:3]].sum(axis=1)
df['total_rech_amt_monthly'] = df.loc[:,rech_num_amt[3:]].sum(axis=1)

In [ ]:
df['amt_per_recharge_monthly'] = df['total_rech_amt_monthly'] / df['total_rech_num_monthly'] 

In [ ]:
df.loc[df['amt_per_recharge_monthly'].isnull(),['total_rech_amt_monthly','total_rech_num_monthly']]

In [ ]:
df['amt_per_recharge_monthly'] = df['amt_per_recharge_monthly'].fillna(0)

In [ ]:
df['total_recharge_amount_per_usr'] = df['total_rech_amt_monthly'] 
df['total_recharge_freq_per_usr'] = df['total_rech_num_monthly'] 


In [ ]:
df = df.drop(['total_rech_amt_monthly','total_rech_num_monthly','total_rech_num_6',
 'total_rech_num_7',
 'total_rech_num_8',
 'total_rech_amt_6',
 'total_rech_amt_7',
 'total_rech_amt_8','amt_per_recharge_6', 'amt_per_recharge_7', 'amt_per_recharge_8'],axis=1)

In [ ]:
date_rech = df.columns[df.columns.str.contains(r"date_of_last_rech_\d{1}")]
date_rech

In [ ]:
for i in date_rech :
    df[i] = pd.to_datetime(df[i],format="%m/%d/%Y")

In [ ]:
df.loc[df['date_of_last_rech_6'].isnull(),date_rech]

In [ ]:
recharge_duration=['prev_recharge_duration','next_recharge_duration']
df[recharge_duration] = df[date_rech].diff(axis=1).iloc[:,1:]

In [ ]:
df[recharge_duration].isnull().sum()

In [ ]:
df.loc[df['prev_recharge_duration'].isnull(),recharge_duration]

In [ ]:
df = df.drop(columns = date_rech)

In [ ]:
df[recharge_duration] = df[recharge_duration].apply(lambda x : x.dt.days)

In [ ]:
df.loc[df['prev_recharge_duration'].isnull(),recharge_duration]

In [ ]:
df['prev_recharge_duration'].describe()

In [ ]:
df['next_recharge_duration'].describe()

- Fill in zero because there weren't any recharge in that month for the user

In [ ]:
df['prev_recharge_duration'] = df['prev_recharge_duration'].fillna(0)
df['next_recharge_duration'] = df['next_recharge_duration'].fillna(0)

In [ ]:
df.groupby(['churn_probability'])[['prev_recharge_duration','next_recharge_duration']].describe()

In [ ]:
i = 'prev_recharge_duration'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'next_recharge_duration'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
max_rech_cols = [i for i in recharge_cols if re.search(r"max_rech",i)]
print(max_rech_cols)
df = df.drop(columns=max_rech_cols)

In [ ]:
df = df.drop(['last_day_rch_amt_6',
       'last_day_rch_amt_7', 'last_day_rch_amt_8'],axis=1)

In [ ]:
date_data_rech = df.columns[df.columns.str.contains(r"date_of_last_rech_data_\d{1}")]
date_data_rech 

### To many null values
- drop columns
- A lot of customers don't seem to opt for data

In [ ]:
df[date_data_rech].isnull().sum()/ len(df)

In [ ]:
df = df.drop(date_data_rech,axis=1)

In [ ]:
[i for i in df.columns if re.search(r'rech',i)]

In [ ]:
rech_data =  [i for i in recharge_cols if re.search(r"total_rech_data",i)]
rech_data

In [ ]:
df[rech_data].isnull().sum() / len(df)

In [ ]:
# df['data_opted'] = np.where(df[rech_data].isnull().sum(axis=1) < 3 ,1,0)

In [ ]:
# df = df.drop(rech_data,axis=1)

In [ ]:
count_rech_data = [i for i in df.columns if re.search(r'^count_rech_[2-3]g',i)]
count_rech_data

In [ ]:
avg_rech_data = [i for i in df.columns if re.search(r'^av_rech_amt',i)]
avg_rech_data

In [ ]:
df[avg_rech_data] = df[avg_rech_data].fillna(0)
df[rech_data] = df[rech_data].fillna(0)

In [ ]:
df = df.drop(columns=['count_rech_2g_6', 'count_rech_2g_7',
       'count_rech_2g_8', 'count_rech_3g_6', 'count_rech_3g_7',
       'count_rech_3g_8'])

In [ ]:
df[rech_data].sum()

In [ ]:
df[rech_data].describe(np.linspace(.75,1,25))

In [ ]:
df['data_per_recharge_monthly'] = (df[avg_rech_data].fillna(0).sum(axis=1) / df[rech_data].fillna(0).sum(axis=1)).fillna(0)

In [ ]:
for amt,freq in zip(avg_rech_data,rech_data) : 
     df[f'data_per_recharge_'+ amt.split('_')[-1]] = df[amt] / df[freq]

In [ ]:
data_per_recharge  = [i for i in df.columns if re.search(r"data_per_recharge_[6-8]?",i)]
data_per_recharge

In [ ]:
df.groupby(['churn_probability'])[data_per_recharge].describe().T

In [ ]:
df['data_per_recharge_diff_7_6'] = df['data_per_recharge_7'].fillna(0) - df['data_per_recharge_6'].fillna(0)
df['data_per_recharge_diff_8_7'] = df['data_per_recharge_8'].fillna(0) - df['data_per_recharge_7'].fillna(0)

In [ ]:
df['data_per_recharge_phase_drift'] = df['data_per_recharge_8'].fillna(0) - ((df['data_per_recharge_7'].fillna(0) + df['data_per_recharge_6'].fillna(0))/2)

In [ ]:
i = 'data_per_recharge_diff_7_6'
df2 = df.loc[(df[i] > df[i].quantile(.05)) & (df[i] < df[i].quantile(.95))]
# df2 = df.copy()
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'data_per_recharge_diff_8_7'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'data_per_recharge_phase_drift'
df2 = df.loc[(df[i] > df[i].quantile(.1)) & (df[i] < df[i].quantile(.9))]
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
data_drop = avg_rech_data+rech_data
data_drop

In [ ]:
df = df.drop(columns = data_drop)

In [ ]:
df[['og_others_6', 'og_others_7', 'og_others_8', 'ic_others_6',
       'ic_others_7', 'ic_others_8']].describe()

In [ ]:
df = df.drop(columns= ['og_others_6', 'og_others_7', 'og_others_8', 'ic_others_6',
       'ic_others_7', 'ic_others_8'])

In [ ]:
[i for i in df.columns if re.search(r'rech',i)]

In [ ]:
df = df.drop( ['data_per_recharge_6',
 'data_per_recharge_7',
 'data_per_recharge_8'],axis=1)

# Vol Cols

In [ ]:
vol_cols = [i for i in df.columns if re.search(r"vol_",i)]
vol_cols

In [ ]:
order_list = []
for j in [6,7,8] : 
    order_list.append([i for i in vol_cols if re.search(r'vol_[2-3]g_mb_{}'.format(j),i)])
order_list

In [ ]:
for i,j in zip(order_list,[6,7,8]) : 
    df['vol_mb_{}'.format(j)] = df[i].sum(axis=1)

In [ ]:
df = df.drop(vol_cols,axis=1)

In [ ]:
vol_cols = [i for i in df.columns if re.search(r"vol_",i)]
vol_cols

In [ ]:
df.groupby(['churn_probability'])[vol_cols].describe().T

In [ ]:
df['vol_mb_diff_7_6'] = df['vol_mb_7'] - df['vol_mb_6']
df['vol_mb_diff_8_7'] = df['vol_mb_8'] - df['vol_mb_7']


In [ ]:
df['vol_mb_phase_drift'] = df['vol_mb_8']  - ((df['vol_mb_7'] + df['vol_mb_6'])/2)

In [ ]:
df['vol_mb_monthly'] = df[vol_cols].sum(axis=1) / len(vol_cols)

In [ ]:
df['mobile_data_usg_per_usr'] = df[vol_cols].sum(axis=1)

In [ ]:
i = 'vol_mb_diff_7_6'
df2 = df.loc[(df[i] > df[i].quantile(.75)) & (df[i] < df[i].quantile(.9))]
# df2 = df.copy()
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'vol_mb_diff_8_7'
df2 = df.loc[(df[i] > df[i].quantile(.75)) & (df[i] < df[i].quantile(.9))]
# df2 = df.copy()
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
i = 'vol_mb_phase_drift'
df2 = df.loc[(df[i] > df[i].quantile(.75)) & (df[i] < df[i].quantile(.9))]
# df2 = df.copy()
sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,color='b')
sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,color='r')

In [ ]:
df = df.drop(vol_cols,axis=1)

### Monthly Schemes

In [ ]:
monthly_schemes = df.columns[df.columns.str.contains(r"^monthly")]
{i:df[i].unique() for i in monthly_schemes}

In [ ]:
monthly_6 = [i for i in monthly_schemes if re.search(r"_6$",i)]
monthly_7 = [i for i in monthly_schemes if re.search(r"_7$",i)]
monthly_8 = [i for i in monthly_schemes if re.search(r"_8$",i)]

In [ ]:
df['monthly_6'] = df[monthly_6].sum(axis=1)
df['monthly_7'] = df[monthly_7].sum(axis=1)
df['monthly_8'] = df[monthly_8].sum(axis=1)

In [ ]:
df = df.drop(monthly_6 + monthly_7 + monthly_8,axis=1)

In [ ]:
monthly_schemes = df.columns[df.columns.str.contains(r"^monthly")]
monthly_schemes

In [ ]:
df.groupby('churn_probability')[monthly_schemes].describe().T

In [ ]:
df['total_monthly_schemes_per_usr'] = df[monthly_schemes].sum(axis=1) 

### Sachet Schemes

In [ ]:
sachet_schemes = df.columns[df.columns.str.contains(r"^sachet")]
{i:df[i].unique() for i in sachet_schemes}

In [ ]:
sachet_6 = [i for i in sachet_schemes if re.search(r"_6$",i)]
sachet_7 = [i for i in sachet_schemes if re.search(r"_7$",i)]
sachet_8 = [i for i in sachet_schemes if re.search(r"_8$",i)]

In [ ]:
df['sachet_6'] = df[sachet_6].sum(axis=1)
df['sachet_7'] = df[sachet_7].sum(axis=1)
df['sachet_8'] = df[sachet_8].sum(axis=1)

In [ ]:
df = df.drop(sachet_6 + sachet_7 + sachet_8,axis=1)
sachet_schemes = df.columns[df.columns.str.contains(r"^sachet")]
sachet_schemes

In [ ]:
df.groupby('churn_probability')[sachet_schemes].describe().T

In [ ]:
df['total_sachet_schemes_per_usr'] = df[sachet_schemes].sum(axis=1) 

In [ ]:
binary_schemes = df.columns[df.columns.str.contains(r"^(fb|night)")]
{i:df[i].unique() for i in binary_schemes}

In [ ]:
df[binary_schemes].isnull().sum()

- It seems that night_pck_users and fb_users are only eligible for these schemes if they have either of monthly and sachet schemes
- This is also seems to be in-line with nulls in the mobile data related columns
- This goes to show that people who didn't opt for data are given nulls in the mobile data related cols
- Lets create a missing indicator column for identifying if the customer opted for null or not
- Since we will be using ensemble techniques (decision trees) , there is no need to one-hot encode the columns

In [ ]:
df.loc[df['night_pck_user_6'].notnull(),['monthly_6','sachet_6']].sum(axis=1).astype('bool').unique()

In [ ]:
df.loc[df['night_pck_user_6'].isnull(),['monthly_6','sachet_6']].sum(axis=1).astype('bool').unique()

In [ ]:
df.loc[df['fb_user_7'].notnull(),['monthly_7','sachet_7']].sum(axis=1).astype('bool').unique()

In [ ]:
df.loc[df['fb_user_7'].isnull(),['monthly_7','sachet_7']].sum(axis=1).astype('bool').unique()

In [ ]:
df['missing_indicator_mobile_data_jun'] = np.where(df['night_pck_user_6'].isnull(),1,0)
df['missing_indicator_mobile_data_jul'] = np.where(df['night_pck_user_7'].isnull(),1,0)
df['missing_indicator_mobile_data_aug'] = np.where(df['night_pck_user_8'].isnull(),1,0)

In [ ]:
df[binary_schemes] = df[binary_schemes].fillna(0)

In [ ]:
df[['aug_vbc_3g',
       'jul_vbc_3g', 'jun_vbc_3g']].isnull().sum()

In [ ]:
df.groupby(['churn_probability'])[['aug_vbc_3g',
       'jul_vbc_3g', 'jun_vbc_3g']].describe().T

In [ ]:
vbc_monthly = ['aug_vbc_3g',
       'jul_vbc_3g', 'jun_vbc_3g']
df['total_vbc_per_usr'] = df[vbc_monthly].sum(axis=1)
df['vbc_monthly'] = df['total_vbc_per_usr'] / len(vbc_monthly)

In [ ]:
df['vbc_diff_7_6'] = df['jul_vbc_3g'] - df['jun_vbc_3g']
df['vbc_diff_8_7'] = df['aug_vbc_3g'] - df['jul_vbc_3g']

In [ ]:
df = df.drop(columns=['aug_vbc_3g',
       'jul_vbc_3g', 'jun_vbc_3g'])

In [ ]:
df.columns

### Null Values Imputated
- Where ever applicable missing indicator was used to identify missing values , since there were some features with MAR values 
- There were some where zero imputation was applicabled

In [ ]:
display(df.isnull().sum().to_dict())

In [ ]:
df.shape

### Deal with Outliers

In [ ]:
def get_dist_box(df,col_kind,x,y):
    cols = 2
    rows = np.ceil(len(col_kind)/cols).astype('int')
    fig,ax=plt.subplots(rows,cols,figsize=(x,y))
    for idx,i in enumerate(col_kind) : 
        idx1 = idx // cols
        idx2 = idx % cols
        if len(list(ax.shape)) !=1 : 
            sns.boxplot(data=df,y=i,x='churn_probability',ax=ax[idx1,idx2])
            ax[idx1,idx2].set_title(f'{i}')
        else : 
            sns.boxplot(data=df,y=i,x='churn_probability',ax=ax[idx2])
            ax[idx2].set_title(f'{i}')
    [fig.delaxes(i) for i in ax.flat if not i.has_data()]
    plt.tight_layout()
    plt.show()

In [ ]:
def quantile_plot(df,col_kind,qrange = np.arange(0,1.1,0.1),label_range=np.arange(0,110,10),x=0,y=0): 
    quantile_df = df[col_kind].quantile(list(qrange)).reset_index().rename(columns = {'index' : 'quantile'})
    cols = 2
    rows = np.ceil(len(col_kind)/cols).astype('int')
    fig,ax=plt.subplots(rows,cols,figsize=(x,y))
    for idx,i in enumerate(col_kind) : 
        idx1 = idx // cols
        idx2 = idx % cols
        if len(list(ax.shape)) !=1 : 
            sns.pointplot(data=quantile_df,x='quantile',y=i,ax=ax[idx1,idx2])
            ax[idx1,idx2].set_title(f'percentile Vs {i}')
            ax[idx1,idx2].set_xticklabels(rotation=45,labels = list(label_range))
        else : 
            sns.pointplot(data=quantile_df,x='quantile',y=i,ax=ax[idx2])
            ax[idx2].set_title(f'percentile Vs {i}')
            ax[idx2,].set_xticklabels(rotation=45,labels = list(label_range))
    [fig.delaxes(i) for i in ax.flat if not i.has_data()]
    plt.tight_layout()
    plt.show()

In [ ]:
def cap_outliers_iqr(df,left_limit,right_limit) : 
    return right_limit if df > right_limit else left_limit
def iqr_treatment_revised(df,col_kind,lq=.25,rq=.75) : 
#     df = df[df['churn_probability'] == 0]
    cap = {}
    for i in col_kind : 
        q1 = df[i].quantile(lq)
        q3 = df[i].quantile(rq)
        iqr = q3 - q1 
        left_limit = q1 - 1.5 * iqr
        right_limit = q3 + 1.5 * iqr
        df[i] = df[i].apply(lambda x : x if left_limit<=x<=right_limit or pd.isnull(x) else cap_outliers_iqr(x,left_limit,right_limit))
        cap[i] = [left_limit,right_limit]
#         print(df.shape)
    return df,cap

In [ ]:
def cap_outliers_z(df,left_limit,right_limit) : 
    return right_limit if df > right_limit else left_limit
def z_treatment_revised(df,col_kind,lq=.25,rq=.75,factor=3) : 
#     df = df[df['churn_probability'] == 0]
    cap = {}
    for i in col_kind : 
        x = df[i].mean()
        s = df[i].quantile(rq)
        left_limit = x - factor * s
        right_limit = x + factor * s 
        df[i] = df[i].apply(lambda x : x if left_limit<=x<=right_limit or pd.isnull(x) else cap_outliers_z(x,left_limit,right_limit))
        cap[i] = [left_limit,right_limit]
#         print(df.shape)
    return df,cap

In [ ]:
df.columns

In [ ]:
monthly_cols = [i for i in df.columns if re.search(r'monthly$',i)]
diff_cols = [i for i in df.columns if re.search(r'diff',i)]
per_usr = [i for i in df.columns if re.search(r'_per_usr$',i)]
phase_cols = [i for i in df.columns if re.search(r'_phase_drift$',i)]

In [ ]:
monthly_cols

In [ ]:
diff_cols

In [ ]:
phase_cols

In [ ]:
per_usr = [i for i in per_usr if i not in ['total_monthly_schemes_per_usr',
 'total_sachet_schemes_per_usr']]
per_usr

### Monthly Col Outlier Analysis

- It seems there are a lot of outliers in the Monthly Columns

In [ ]:
get_dist_box(df.copy(),monthly_cols,10,15)

- Most of the outliers seem to lie at the 99th quantile on granular analysis

In [ ]:
 quantile_plot(df,monthly_cols,np.arange(0,1.1,0.1),np.arange(0,110,10),x=10,y=20)

In [ ]:
 quantile_plot(df,monthly_cols,np.arange(0.9,1.01,0.01),np.arange(90,101,1),x=10,y=20)

In [ ]:
df2,cap_iqr_monthly = iqr_treatment_revised(df.copy(),monthly_cols,.1,.9)
get_dist_box(df2,monthly_cols,10,15)

In [ ]:
df3,cap_z_monthly= z_treatment_revised(df.copy(),monthly_cols,.1,.9,3)
get_dist_box(df3,monthly_cols,10,15)

In [ ]:
before_capping = df.copy()

In [ ]:
df = df2

In [ ]:
chosen_cap_monthly = cap_iqr_monthly

### Diff Col Outlier Analysis

In [ ]:
get_dist_box(df.copy(),diff_cols,10,45)

In [ ]:
quantile_plot(df,diff_cols,np.arange(0,1.1,0.1),np.arange(0,110,10),x=10,y=50)

In [ ]:
quantile_plot(df,diff_cols,np.arange(.9,1.01,0.01),np.arange(90,101,1),x=10,y=50)

In [ ]:
quantile_plot(df,diff_cols,np.arange(0.,.11,0.01),np.arange(0,11,1),x=10,y=50)

In [ ]:
df2,cap_iqr_diff = iqr_treatment_revised(df.copy(),diff_cols,.1,.9)
get_dist_box(df2,diff_cols,10,45)

In [ ]:
df3,cap_z_diff= z_treatment_revised(df.copy(),diff_cols,.1,.9,4)
get_dist_box(df3,diff_cols,10,45)

In [ ]:
chosen_cap_diff = cap_iqr_diff
df = df2

### Per User  Col Outlier Analysis 

In [ ]:
get_dist_box(df.copy(),per_usr,10,10)

In [ ]:
quantile_plot(df,per_usr,np.arange(0,1.1,0.1),np.arange(0,110,10),x=10,y=15)

In [ ]:
df2,cap_iqr_usr = iqr_treatment_revised(df.copy(),per_usr,.1,.9)
get_dist_box(df2,per_usr,10,15)

In [ ]:
df3,cap_z_usr= z_treatment_revised(df.copy(),per_usr,.1,.9,4)
get_dist_box(df3,per_usr,10,15)

In [ ]:
chosen_cap_usr = cap_z_usr
df = df3

### Phase Drift Cols Outlier Analysis

In [ ]:
get_dist_box(df.copy(),phase_cols,10,15)

In [ ]:
quantile_plot(df,phase_cols,np.arange(0,1.1,0.1),np.arange(0,110,10),x=10,y=20)

In [ ]:
quantile_plot(df,phase_cols,np.arange(0.9,1.01,0.01),np.arange(90,101,1),x=10,y=20)

In [ ]:
df2,cap_iqr_phase = iqr_treatment_revised(df.copy(),phase_cols,.1,.9)
get_dist_box(df2,phase_cols,10,15)

In [ ]:
df3,cap_z_phase= z_treatment_revised(df.copy(),phase_cols,.1,.9,3)
get_dist_box(df3,phase_cols,10,15)

In [ ]:
chosen_cap_phase = cap_z_phase
df = df3

In [ ]:
cat_cols = ['monthly_6', 'monthly_7', 'monthly_8',
       'total_monthly_schemes_per_usr', 'sachet_6', 'sachet_7', 'sachet_8',
       'total_sachet_schemes_per_usr']

In [ ]:
# These columns have special circumstances , it is their nature to be immensely right skewed 
df[cat_cols].describe()

In [ ]:
per_usr = [i for i in df.columns if re.search(r'_per_usr$',i)]
per_usr

In [ ]:
for i in per_usr : 
    ext = re.sub(r'(total_|_per_usr)','',i)
    df[ext + '_rate'] = df[i] / df['aon']
    


In [ ]:
df = df.drop(per_usr,axis=1)

In [ ]:
rate_cols = [i for i in df.columns if re.search(r'_rate$',i)]
rate_cols

In [ ]:
def rate_col_prob_dist(df2) : 
    cols = 2
    rows = np.ceil(len(rate_cols)/cols).astype('int')
    fig,ax=plt.subplots(rows,cols,figsize=(15,25))
    for idx,i in enumerate(rate_cols) : 
        idx1 = idx // cols
        idx2 = idx % cols
        df2 = df.loc[(df[i] > df[i].quantile(.05)) & (df[i] < df[i].quantile(.95))]
#         ax2 = ax[idx1,idx2].twinx()
        sns.kdeplot(data=df2[df2.churn_probability == 0],x=i,ax=ax[idx1,idx2],color='b')
        sns.kdeplot(data=df2[df2.churn_probability == 1],x=i,ax=ax[idx1,idx2],color='r')
        ax[idx1,idx2].set_title(f'{i}')
    [fig.delaxes(i) for i in ax.flat if not i.has_data()]
    plt.tight_layout()
    plt.legend('01')
    plt.show()
rate_col_prob_dist(df)

In [ ]:
def bins_churn_rate_charts(df2,kind_cols,q,x,y) : 
    cols = 2
    rows = np.ceil(len(kind_cols)/cols).astype('int')
    fig,ax=plt.subplots(rows,cols,figsize=(x,y))
    for idx,i in enumerate(kind_cols) : 
#         print(i)
        idx1 = idx // cols
        idx2 = idx % cols
        df2[i + '_bins']= pd.qcut(df2[i],q=q,duplicates='drop',precision=2)
        bin_df = df2.groupby([i+'_bins'])['churn_probability'].sum().reset_index()
        bin_df["churn_percent"] = bin_df['churn_probability']/ bin_df['churn_probability'].sum()
        sns.barplot(data=bin_df,x=i+'_bins',y='churn_percent',ax=ax[idx1,idx2])
        ax[idx1,idx2].set_title(f'{i}')
        ax[idx1,idx2].set_xticks(ax[idx1,idx2].get_xticks())
        ax[idx1,idx2].tick_params(axis='x', rotation=90)
#         ax[idx1,idx2].set_xticklabels(ax[idx1,idx2].get_xticks(),rotation=90)
    [fig.delaxes(i) for i in ax.flat if not i.has_data()]
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(df[rate_cols + ['churn_probability']].corr(method='spearman'),annot=True)
plt.show()

In [ ]:
df['total_mins_incoming_outgoing'] = df['incoming_mins_usg_rate'] + df['incoming_mins_usg_rate']

In [ ]:
bins_churn_rate_charts(df.copy(),rate_cols,5,15,25)

In [ ]:
# It seems the rate cols are not of much use drop them 
df = df.drop(columns = rate_cols,axis=1)

- Negative Diff has higher churn rates
- Also the diff between 8th and 7th months captures churn rate bettern than diff between the 7th and the 6th month
- This is expected because customers showing lower activity towards the end is an expected attribute of churn

In [ ]:
bins_churn_rate_charts(df.copy(),diff_cols,5,15,65)

- It is clear at a glance that as you move from higher bins to lower bins the churn rate increases
- The only expection to the rule here seems to be std calls

In [ ]:
bins_churn_rate_charts(df.copy(),monthly_cols,5,10,25)

In [ ]:
bins_churn_rate_charts(df.copy(),phase_cols,5,10,30)

In [ ]:
cat_cols = [i for i in cat_cols if not i in ['total_monthly_schemes_per_usr','total_sachet_schemes_per_usr'] ]

- Higher the noof schemes the better a person is enrolled in the better

In [ ]:
df2 =df.copy()
rows = np.ceil(len(cat_cols)/cols).astype('int')
fig,ax=plt.subplots(rows,cols,figsize=(10,15))
for idx,i in enumerate(cat_cols) : 
#         print(i)
    idx1 = idx // cols
    idx2 = idx % cols
    bin_df = df2.groupby([i])['churn_probability'].sum().reset_index()
    bin_df["churn_percent"] = bin_df['churn_probability']/ bin_df['churn_probability'].sum()
    sns.barplot(data=bin_df,x=i,y='churn_percent',ax=ax[idx1,idx2])
    ax[idx1,idx2].set_title(f'{i}')
    ax[idx1,idx2].set_xticks(ax[idx1,idx2].get_xticks())
    ax[idx1,idx2].tick_params(axis='x', rotation=90)
#         ax[idx1,idx2].set_xticklabels(ax[idx1,idx2].get_xticks(),rotation=90)
[fig.delaxes(i) for i in ax.flat if not i.has_data()]
plt.tight_layout()
plt.show()

In [ ]:
# As the duration of calls increases the churn rate decreases
df2 =df.copy()
df2['ic_og_mins_bins'] = pd.qcut(df2['total_mins_incoming_outgoing'],q=5)
bins_df = df2.groupby(['ic_og_mins_bins'])['churn_probability'].mean().reset_index()
bins_df['perc_churn'] = bins_df['churn_probability'] / bins_df['churn_probability'].sum()
sns.barplot(data=bins_df,x='ic_og_mins_bins',y='perc_churn')
plt.xticks(rotation=90)
plt.show()

### Identifying High Value Customers

In [ ]:
pd.qcut(df['total_arpu_monthly'],q=5).value_counts(sort=False)

In [ ]:
pd.qcut(df['amt_per_recharge_monthly'],q=5).value_counts(sort=False)

In [ ]:
arpu_l = 272
arpu_r=1624
rech_l=47
rech_r=219

In [ ]:
df['high_value_customer']= np.where(df['total_arpu_monthly'].between(arpu_l,arpu_r) & df['amt_per_recharge_monthly'].between(rech_l,rech_r),1,0)

In [ ]:
df['high_value_customer'].value_counts()

- So the according to the above condition there seem to be 18,304 high value customers

In [ ]:
get_dist_box(df,cat_cols+['total_mins_incoming_outgoing'],10,15)

In [ ]:
df2,cap_iqr_cat = iqr_treatment_revised(df.copy(),cat_cols+['total_mins_incoming_outgoing'],.1,.9)
get_dist_box(df2,cat_cols,10,15)


In [ ]:
df3,cap_z_cat= z_treatment_revised(df.copy(),cat_cols+['total_mins_incoming_outgoing'],.1,.9,5)
get_dist_box(df3,cat_cols,10,15)

In [ ]:
chosen_cap_cat = cap_iqr_cat
df = df2

In [ ]:
df = df.drop(columns = 'arpu_avg_7_6')

In [ ]:
y = df.pop('churn_probability')
high_value_customer = df.pop('high_value_customer')
idc = df.pop('id')
X = df.copy()

In [ ]:
# idc = idc.reset_index(drop=True)
# high_value_customer = high_value_customer.reset_index(drop=True)

In [ ]:
missing_indicators = [i for i in df.columns if re.search(r'^(missing_indicator|night|fb|total_mins)',i)]
numeric_cols = np.setdiff1d(df.columns,missing_indicators)

In [ ]:
missing_indicators

In [ ]:
X_train,X_test,y_train,y_test  = train_test_split(X,y,train_size=0.8, random_state=100)

In [ ]:
X_test_or = X_test.copy()
y_test_or = y_test.copy()

In [ ]:
X_test_or.index = X_test.index
y_test_or.index = y_test.index

### Training 

In [ ]:

counter = Counter(y_train)
print(counter)
# transform the dataset
oversample = SMOTE()
X_train, y_train = oversample.fit_resample(X_train, y_train)
# summarize the new class distribution
counter = Counter(y_train)
print(counter)

In [ ]:
scaler = MinMaxScaler()
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
X_train_cat = X_train[missing_indicators]
X_train = X_train[numeric_cols]
X_test_cat = X_test[missing_indicators]
X_test = X_test[numeric_cols]
X_train2 = scaler.fit_transform(X_train)
X_test2 = scaler.transform(X_test )
X_train = pd.DataFrame(X_train2,columns=X_train.columns)
X_test = pd.DataFrame(X_test2,columns=X_train.columns)
X_train = pd.concat([X_train,X_train_cat],axis=1)
X_test = pd.concat([X_test,X_test_cat],axis=1)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
def classification_metrics(y_test,probability,tr) : 
    print(tr)
    y_pred = pd.Series(probability).apply(lambda x : 1 if x >=tr else 0)
    tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
    accuracy = (tp + tn) / (tp + tn + fn + fp)
    specificity = tn / (tn+fp)
    senstivity = tp / (tp+fn)
    precision = tp / (tp+fp)
    fpr = 1 - specificity
    f1_score = 2 * (senstivity * precision) / (senstivity + precision)
    print("True Positives : ",tp)
    print("False Negatives : " , fn)
    print("True Negatives : ",tn)
    print("False Positives : ",fp)
    print("Accuracy : ",accuracy)
    print("Specificity : " , specificity)
    print ("Senstivity : ",senstivity)
    print("Precision : ",precision)
    print("False Positive Rate : ", fpr )
    return tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred
def roc_auc_plot(senstivity_list,specificity_list) : 
    tpr_list = senstivity_list
    fpr_list = [ 1 - i for i in specificity_list]
    sns.lineplot(y=tpr_list,x=fpr_list,color='g')
    sns.lineplot(x=[0,1],y=[0,1],lw=2,linestyle='--',color='r')
    plt.show()
    auc_score = auc(fpr_list,tpr_list)
    print("Area Under the Curve : {0}".format(auc_score))
    
def threshold_plot(accuracy,pred_dir1,pred_dir2,kind) :
    metric1,metric2 = ['specificity','senstivity'] if kind == 'specificity' else ['precision','recall']
    x = np.arange(0,1.1,.1)
    fig,ax = plt.subplots(1,1)
    sns.lineplot(y=accuracy,x=x,color='r',ax=ax,label='accuracy')
    sns.lineplot(y=pred_dir1,x=x,color='b',ax=ax,label=metric1)
    sns.lineplot(y=pred_dir2,x=x,color='g',ax=ax,label=metric2)
    plt.legend(loc='upper right',bbox_to_anchor=(1.4,1))
    plt.xlabel('threshold')
    plt.ylabel('metric value')
    plt.show()
    
def threshold_lists(y_test,probability) : 
    accuracy_list = []
    specificity_list = []
    senstivity_list = []
    precision_list = []
    for tr in np.arange(0,1.1,.1) : 
        y_pred = pd.Series(probability).apply(lambda x : 1 if x >=tr else 0)
        tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
        accuracy = (tp + tn) / (tp + tn + fn + fp)
        specificity = tn / (tn+fp)
        senstivity = tp / (tp+fn)
        precision = tp / (tp+fp)
        accuracy_list.append(accuracy)
        specificity_list.append(specificity)
        senstivity_list.append(senstivity)
        precision_list.append(precision)
    return accuracy_list,specificity_list,senstivity_list,precision_list
    
        
def Model_Ensemble(mdl,X_train,y_train,X_test) : 
    mdl.fit(X_train,y_train)
    y_prob = mdl.predict_proba(X_test)
    churn_probs = y_prob[:,1]
    return mdl,churn_probs
    

### Random Forest Classifier

In [ ]:
rf = RandomForestClassifier()
rf,probability = Model_Ensemble(rf,X_train,y_train,X_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,0.5)

In [ ]:
fimp = pd.Series(dict(zip(X_train.columns,rf.feature_importances_))).sort_values(ascending=False)

In [ ]:
fimp

In [ ]:
fimp = fimp[fimp.cumsum().diff()[(fimp.cumsum().diff() > .01) |(fimp.cumsum().diff().isnull()) ].index]

In [ ]:
fimp

In [ ]:
plt.figure(figsize=(15,5))
sns.pointplot(x=fimp.index,y=fimp,color='r')
plt.xticks(rotation=90)
plt.axhline(0.04,color='b')
plt.show()

### Xgboost

In [ ]:
xgb = XGBClassifier()
xgb,probability = Model_Ensemble(xgb,X_train,y_train,X_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,0.5)

In [ ]:
fimp = pd.Series(dict(zip(X_train.columns,rf.feature_importances_))).sort_values(ascending=False)

In [ ]:
fimp = fimp[fimp.cumsum().diff()[(fimp.cumsum().diff() > .01) |(fimp.cumsum().diff().isnull()) ].index]

In [ ]:
fimp

In [ ]:
plt.figure(figsize=(15,5))
sns.pointplot(x=fimp.index,y=fimp,color='r')
plt.xticks(rotation=90)
plt.axhline(0.04,color='b')
plt.show()

In [ ]:
# def rfe_method(X_train,y_train,n) : 
#     estimator  = RandomForestClassifier()
#     rfe_feature_selection  = RFE(estimator, n_features_to_select=n, step=1)
#     rfe_feature_selection = rfe_feature_selection.fit(X_train,y_train)
#     rankings = rfe_feature_selection.ranking_
#     support  = rfe_feature_selection.support_
#     rfe_df = pd.DataFrame(data= {'features' : X_train.columns,'support' : support,'ranking' : rankings,})
#     return rfe_df.sort_values(['ranking']),support

In [ ]:
# rfe_method(X_train,y_train,40)

In [ ]:
# col_list = X_train.columns[[ True,  True,  True,  True,  True,  True,  True,  True,  True,
#          True,  True,  True,  True,  True,  True, False, False,  True,
#          True,  True,  True,  True,  True,  True,  True,  True, False,
#          True,  True,  True,  True,  True,  True,  True,  True,  True,
#          True,  True,  True,  True,  True,  True, False,  True,  True,
#         False, False, False,  True,  True,  True, False, False,  True,
#         False, False, False]]

In [ ]:
# col_list = fimp[fimp < .93].index

### Principal Component Analysis - Dimensionality Reduction

In [ ]:
pca = PCA()
pca.fit(X_train[numeric_cols])

In [ ]:
skree_df = pd.DataFrame(pca.explained_variance_ratio_).cumsum()
skree_df

In [ ]:
plt.plot(skree_df)
plt.axhline(0.991,color='r')
plt.show()

In [ ]:
skree_df[skree_df[0] < .991].index

In [ ]:
pca = PCA(n_components=42)
pca_train = pd.DataFrame(pca.fit_transform(X_train[numeric_cols]))
pca_test = pd.DataFrame(pca.transform(X_test[numeric_cols]))

In [ ]:
pca_train.columns = ["pc_" + str(i) for i in range(1,43)]
pca_test.columns = ["pc_" + str(i) for i in range(1,43)]

In [ ]:
pca_train = pd.concat([pca_train , X_train[missing_indicators]],axis=1)
pca_test = pd.concat([pca_test , X_test[missing_indicators]],axis=1)

In [ ]:
pca_train.shape,pca_test.shape

### Random Forest GridSearchCV

In [ ]:
rf = RandomForestClassifier()

In [ ]:
# folds = KFold(n_splits=5,shuffle=True,random_state=100)
# parameters  = {'n_estimators' : [50,70], 'criterion' : ['gini'],'max_depth' : [30,50,80],'min_samples_split' : [80,90,100,120],'min_samples_leaf' : [30,40,50,60]}
# crval_rf = GridSearchCV(estimator=rf,scoring=['accuracy','precision','recall','roc_auc'],
#              cv=folds,param_grid = parameters,return_train_score=True,verbose=3,refit='recall')

In [ ]:
# crval_rf.fit(pca_train,y_train)

In [ ]:
# crval_rf.best_params_

Hyperparameters roc score refit: {'criterion': 'gini',
 'max_depth': 50,
 'min_samples_leaf': 30,
 'min_samples_split': 80,
 'n_estimators': 70}

Hyperparameters recall score refit: {'criterion': 'gini',
 'max_depth': 80,
 'min_samples_leaf': 30,
 'min_samples_split': 80,
 'n_estimators': 50}

### ROC Best Params

In [ ]:
rf = RandomForestClassifier(criterion = 'gini',
 max_depth= 50,
 min_samples_leaf =  30,
 min_samples_split = 80,
 n_estimators= 70)

In [ ]:
rf,probability = Model_Ensemble(rf,X_train,y_train,X_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.36)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

In [ ]:
rf,probability = Model_Ensemble(rf,pca_train,y_train,pca_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.39)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

### Recall Best Params 

### Choosen Model of Upgrad Submission
- Has aligned Accuracy , Senstivity and Specificity , to reasonable values at threshold .36
- Since the main goal is to predict high value customers that have churned , having a higher recall gives better results
- Roc Curve Score .92

In [ ]:
rf = RandomForestClassifier(criterion = 'gini',
 max_depth= 80,
 min_samples_leaf =  30,
 min_samples_split = 80,
 n_estimators= 50)

In [ ]:
rf,probability = Model_Ensemble(rf,X_train,y_train,X_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.36)

In [ ]:
# tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.35)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

In [ ]:
rf,probability = Model_Ensemble(rf,pca_train,y_train,pca_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.39)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

### Xgboost GridSearchCV

In [ ]:
xgb = XGBClassifier()
folds = KFold(n_splits=5,shuffle=True,random_state=100)
parameters  = {'criterion' : ['binary:logistic'],'gamma' : [0.01,0.1,1,10,100],'eta' : [0.01,0.1,1],'lambda' : [0.01,1,10,100],'scale_pos_weights' : [1, 2, 5, 10, 20, 50, 100]
}
crval_xgb = GridSearchCV(estimator=xgb,scoring=['accuracy','precision','recall','roc_auc'],
             cv=folds,param_grid = parameters,return_train_score=True,verbose=3,refit='roc_auc')

In [ ]:
# crval_xgb.fit(X_train,y_train)

In [ ]:
# crval_xgb.best_params_

best params intuition based : 0.1,0.1,10,5

 - XGBClassifier Accuracy tuning (criterion = 'binary:logistic' , eta = 1,gamma = 0.1,reg_lambda = 10,scale_pos_weights = 1 )

- XGBClassifier Recall Tuning {'criterion': 'binary:logistic','eta': 1,'gamma': 0.1,'lambda': 10,'scale_pos_weights': 1}

### Accuracy Based Params

### Chosen model for Kaggle Submission - Accuracy Based

- Choosen module for accucary offers highest accuracy currently among all models
- Though the recall is scarificed here , just based on accuracy this is the best model
- Accuracy and Specificity are extremely high in this model
- Also shows the highest precision observed at .7
- ROC Curve Score - 0.85
- Model gives similar results at 0.5 threshold too as seen below 
- Accuracy : 0.925 at .5 threshold and Accuracy : 0.93 at .7 threshold
- The overall best model is the Upgrad Submitted one 
- Choosen 2 thresholds for submission 0.5 and .7

In [ ]:
xgb = XGBClassifier(criterion = 'binary:logistic' , eta = 1,
  gamma = 0.1,
  reg_lambda = 10,
  scale_pos_weights = 1 )

In [ ]:
xgb,probability = Model_Ensemble(xgb,X_train,y_train,X_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.7)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.1)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

In [ ]:
xgb,probability = Model_Ensemble(xgb,pca_train,y_train,pca_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.1)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

### Recall Based Params

In [ ]:
xgb = XGBClassifier(criterion = 'binary:logistic' , eta = 1,
  gamma = 0.1,
  reg_lambda = 100,
  scale_pos_weights = 1 )

In [ ]:
xgb,y_pred = Model_Ensemble(xgb,X_train,y_train,X_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.1)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

In [ ]:
xgb,y_pred = Model_Ensemble(xgb,pca_train,y_train,pca_test)
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)

In [ ]:
threshold_plot(accuracy_list,specificity_list,senstivity_list,'specificity')

In [ ]:
threshold_plot(accuracy_list,precision_list,senstivity_list,'precision')

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.5)

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.1)

In [ ]:
roc_auc_plot(senstivity_list,specificity_list)

### Note :  for each combination of hyperparameters both the models random forest and xgboost have been trained with and without pca . As a general observation , non-pca(models trained without applying data on pca) trained models seemed to better than models with pca applied data . As a result both the submissions are models without dimensionality reduction , this is likely the case because , the features have been generalized to reduce feature set in EDA ,so that there are fewer features which have better discriminatory power

### Test Data

In [ ]:
def test_data() : 
    path = r"data/test.csv"
    df = pd.read_csv(path)

    df = df.drop(columns = ['circle_id'])

    # og_t2o_mou cols 
    drop_og_t2o_mou = df.columns[df.columns.str.contains(r't2o_mou$')]
    df = df.drop(columns = drop_og_t2o_mou)

    arpu_cols = df.columns[df.columns.str.contains(r'arpu')]
    arpu_cols

    # Create total average revenue per user column , do this monthly
    df['total_arpu_6'] = df['arpu_6'] + df['arpu_3g_6'].fillna(0) + df['arpu_2g_6'].fillna(0)
    df['total_arpu_7'] = df['arpu_7'] + df['arpu_3g_7'].fillna(0) + df['arpu_2g_7'].fillna(0)
    df['total_arpu_8'] = df['arpu_8'] + df['arpu_3g_8'].fillna(0) + df['arpu_2g_8'].fillna(0)

    total_arpu_cols = df.columns[df.columns.str.contains(r'^total_arpu_')]
    total_arpu_cols

    df['total_arpu_monthly'] = df[total_arpu_cols].sum(axis=1) / len(total_arpu_cols)

    total_arpu_cols = df.columns[df.columns.str.contains(r'(^total_arpu_)')]
    total_arpu_cols

    df['arpu_diff_7_6'] = df['total_arpu_7'] - df['total_arpu_6']
    df['arpu_diff_8_7'] = df['total_arpu_8'] - df['total_arpu_7']
    df['arpu_diff_8_6'] = df['total_arpu_8'] - df['total_arpu_6']

    df['total_revenue_per_usr'] = df[['total_arpu_6', 'total_arpu_7', 'total_arpu_8']].sum(axis=1)

    df['arpu_avg_7_6'] = (df['total_arpu_7'] + df['total_arpu_6']) /2 


    df['arpu_phase_drift'] = df['total_arpu_8'] - df['arpu_avg_7_6']

    df = df.drop(['total_arpu_6', 'total_arpu_7', 'total_arpu_8','arpu_diff_8_6'],axis=1)

    df = df.drop(['arpu_6', 'arpu_7', 'arpu_8', 'arpu_3g_6', 'arpu_3g_7', 'arpu_3g_8',
           'arpu_2g_6', 'arpu_2g_7', 'arpu_2g_8'],axis=1)

    mou_cols = df.columns[df.columns.str.contains(r'mou')]
    mou_cols

    df['missing_indicator_mins_per_usg_jun'] = np.where(df['onnet_mou_6'].isnull(),1,0)
    df['missing_indicator_mins_per_usg_jul'] = np.where(df['onnet_mou_7'].isnull(),1,0)
    df['missing_indicator_mins_per_usg_aug'] = np.where(df['onnet_mou_8'].isnull(),1,0)

    # Now that there is a missing value indicator fill null values with zero
    df[mou_cols] = df[mou_cols].fillna(0)

    onnet_mou_cols = df.columns[df.columns.str.contains(r"^onnet_mou")]
    offnet_mou_cols = df.columns[df.columns.str.contains(r"^offnet_mou")]

    df['offnet_mou_diff_7_6']= df['offnet_mou_7'] - df['offnet_mou_6']
    df['offnet_mou_diff_8_7'] = df['offnet_mou_8'] - df['offnet_mou_7']

    df['onnet_mou_diff_7_6']= df['onnet_mou_7'] - df['onnet_mou_6']
    df['onnet_mou_diff_8_7'] = df['onnet_mou_8'] - df['onnet_mou_7']

    df['offnet_mou_phase_drift'] = df['offnet_mou_8'] - ((df['offnet_mou_7'] + df['offnet_mou_6']) /2)

    df['onnet_mou_phase_drift'] = df['onnet_mou_8'] - ((df['onnet_mou_7'] + df['onnet_mou_6']) /2)

    df['onnet_mou_monthly'] = df[onnet_mou_cols].sum(axis=1) / len(onnet_mou_cols)
    df['offnet_mou_monthly'] = df[offnet_mou_cols].sum(axis=1) / len(offnet_mou_cols)

    df = df.drop(columns = list(onnet_mou_cols)+list(offnet_mou_cols))

    roam_mou_cols = df.columns[df.columns.str.contains(r"^roam[a-zA-Z_]+mou_[0-9]$")]
    roam_mou_cols

    roam_mou_6 = [i for i in roam_mou_cols if re.search(r"ic|og",i) and re.search(r"6",i)]
    roam_mou_7 = [i for i in roam_mou_cols if re.search(r"ic|og",i) and re.search(r"7",i)]
    roam_mou_8 = [i for i in roam_mou_cols if re.search(r"ic|og",i) and re.search(r"8",i)]
    df['roam_mou_6'] = df[roam_mou_6].sum(axis=1)
    df['roam_mou_7'] = df[roam_mou_7].sum(axis=1)
    df['roam_mou_8'] = df[roam_mou_8].sum(axis=1)

    roam_mou = [i for i in df.columns if re.search(r"^roam_mou_[6-8]$",i)]
    df['roam_mou_monthly'] = df[roam_mou].sum(axis=1) / len(df)

    mou_6 = [i for i in roam_mou_cols if re.search(r'6$',i)]
    mou_7 = [i for i in roam_mou_cols if re.search(r'7$',i)]
    mou_8 = [i for i in roam_mou_cols if re.search(r'8$',i)]


    df['roam_diff_7_6'] = df[mou_7].sum(axis=1) - df[mou_6].sum(axis=1)
    df['roam_diff_8_7'] = df[mou_8].sum(axis=1) - df[mou_7].sum(axis=1)

    df['roam_phase_drift'] = df[mou_8].sum(axis=1) - ((df[mou_7].sum(axis=1) + df[mou_6].sum(axis=1))/2)

    df = df.drop(columns =[i for i in df.columns if re.search(r'roam',i)] )

    def transform_drop_cols(df,loc) : 
        drop_t2_cols = [i for i in df.columns if re.search(r"^{0}_(og|ic)_[a-zA-Z0-9_]+mou_[6-8]$".format(loc),i)]
        df = df.drop(drop_t2_cols,axis=1)

        loc_ig_og = [i for i in df.columns if re.search(r'{0}'.format(loc),i)]
        def extract_num(x) : 
            return x.split('_')[-1]
        mnth_lst = list(set(map(extract_num,loc_ig_og)))

        l = []
        for i in mnth_lst : 
            l.append([col for col in loc_ig_og if re.search(r"{0}_(og|ic)[a-z_]+{1}".format(loc,i),col)])
    #         print(l)
        for m,c in zip(mnth_lst,l) : 
            df[f'{loc}_mou_{m}'] = df[c].sum(axis=1)
    #     df = df.drop(loc_ig_og,axis=1)
        return df,loc_ig_og

    df2 ,loc_ig_og= transform_drop_cols(df.copy(),'loc')
    df2,std_ig_og = transform_drop_cols(df2,'std')
    df2,spl_ig_og = transform_drop_cols(df2,'spl')
    df2,isd_ig_og = transform_drop_cols(df2,'isd')
    df = df2

    mou_cols = [loc_ig_og,
    std_ig_og ,
    spl_ig_og,
    isd_ig_og]

    diff_cols_mou_7_6 = []
    diff_cols_mou_8_7 = []
    phase_drift_mou = []
    def mou_diff_cols(df,mou_cols) :
        prefix = mou_cols[0].split('_')[0]
        mou_6 = [i for i in mou_cols if re.search(r'6$',i)]
        mou_7 = [i for i in mou_cols if re.search(r'7$',i)]
        mou_8 = [i for i in mou_cols if re.search(r'8$',i)]
        df[f'{prefix}_mou_diff_7_6'] = df[mou_7].sum(axis=1) - df[mou_6].sum(axis=1)
        df[f'{prefix}_mou_diff_8_7'] = df[mou_8].sum(axis=1) - df[mou_7].sum(axis=1)
        df[f'{prefix}_mou_phase_drift'] = df[mou_8].sum(axis=1) - ((df[mou_7].sum(axis=1) + df[mou_6].sum(axis=1))/2)
        diff_cols_mou_7_6.append(prefix)
        diff_cols_mou_8_7.append(prefix)
        phase_drift_mou.append(prefix)
        return df

    for cols in mou_cols : 
        df = mou_diff_cols(df,cols)

    diff_cols_mou_7_6 = list(map(lambda x : x + "_mou_diff_7_6" ,diff_cols_mou_7_6))

    diff_cols_mou_8_7 = list(map(lambda x : x + "_mou_diff_8_7" ,diff_cols_mou_8_7))

    loc_cols = df.columns[df.columns.str.contains(r"^loc")]
    df['loc_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

    loc_cols = df.columns[df.columns.str.contains(r"^std")]
    df['std_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

    loc_cols = df.columns[df.columns.str.contains(r"^isd")]
    df['isd_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

    loc_cols = df.columns[df.columns.str.contains(r"^spl")]
    df['spl_mou_monthly'] = df[loc_cols].sum(axis=1) / len(loc_cols)

    mou_monthly_list = df.columns[df.columns.str.contains('_mou_monthly')]


    phase_drift_mou = list(map(lambda x : x + "_mou_phase_drift" ,phase_drift_mou))

    mou_cols = [i for i in df.columns if re.search(r'mou',i)]

    drop_mou = [i for i in  mou_cols if not re.search(r'(mou_monthly$|diff|phase_drift$)',i)]

    drop_date = [i for i in df.columns if re.search(r'^last_date_of_month',i)]
    df = df.drop(drop_date,axis=1)

    total_ic_og  = df.columns[df.columns.str.contains(r'total_(ic|og)_mou')]

    total_ig = [i for i in total_ic_og if re.search(r'ic',i)]
    total_og = [i for i in total_ic_og if re.search(r'og',i)]

    drop_mou = np.setdiff1d(drop_mou,total_ic_og)

    mou_6 = df.columns[df.columns.str.contains(r'^total_(og|ic)_mou_6$')]
    df['total_mou_6'] = df[mou_6].sum(axis=1)
    mou_7 = df.columns[df.columns.str.contains(r'^total_(og|ic)_mou_7$')]
    df['total_mou_7'] = df[mou_7].sum(axis=1)
    mou_8 = df.columns[df.columns.str.contains(r'^total_(og|ic)_mou_8$')]
    df['total_mou_8'] = df[mou_8].sum(axis=1)

    total_mou = df.columns[df.columns.str.contains(r"total_mou_[6-8]$")]
    total_mou

    df['total_incoming_mins_usg_per_usr'] = df[total_ig].sum(axis=1)
    df['total_outgoing_mins_usg_per_usr'] = df[total_og].sum(axis=1)
    df['total_mins_usg_per_usr'] = df[total_mou].sum(axis=1)

    df = df.drop(total_ic_og,axis=1)

    df = df.drop(drop_mou,axis=1)

    df['total_mou_diff_7_6'] = df['total_mou_7'] - df['total_mou_6']
    df['total_mou_diff_8_7'] = df['total_mou_8'] - df['total_mou_7']

    df['total_mou_phase_drift'] = df['total_mou_8'] - ((df['total_mou_7'] + df['total_mou_6'])/2)

    df['total_mou_monthly'] = df[total_mou].sum(axis=1)

    df = df.drop(total_mou,axis=1)

    mou_cols = [i for i in df.columns if re.search(r'mou',i)]
    mou_cols

    recharge_cols = df.columns[df.columns.str.contains(r"rech")]
    recharge_cols

    rech_num_amt = [i for i in recharge_cols if re.search(r"total_rech_(num|amt)",i)]
    rech_num_amt

    df['amt_per_recharge_6'] = df['total_rech_amt_6'] / df['total_rech_num_6']
    df['amt_per_recharge_7'] = df['total_rech_amt_7'] / df['total_rech_num_7']
    df['amt_per_recharge_8'] = df['total_rech_amt_8'] / df['total_rech_num_8']

    amt_per_recharge_mnth = [i for i in df.columns if re.search(r'amt_per_recharge_[6-8]$',i)]

    df[amt_per_recharge_mnth] = df[amt_per_recharge_mnth].fillna(0)

    df['amt_per_recharge_diff_7_6'] = df['amt_per_recharge_7'] - df['amt_per_recharge_6']
    df['amt_per_recharge_diff_8_7'] = df['amt_per_recharge_8'] - df['amt_per_recharge_7']

    df['amt_per_recharge_phase_drift'] = df['amt_per_recharge_8'] - ((df['amt_per_recharge_7'] + df['amt_per_recharge_6'])/2)

    df['total_rech_num_monthly'] = df.loc[:,rech_num_amt[:3]].sum(axis=1)
    df['total_rech_amt_monthly'] = df.loc[:,rech_num_amt[3:]].sum(axis=1)

    df['amt_per_recharge_monthly'] = df['total_rech_amt_monthly'] / df['total_rech_num_monthly'] 

    df['amt_per_recharge_monthly'] = df['amt_per_recharge_monthly'].fillna(0)

    df['total_recharge_amount_per_usr'] = df['total_rech_amt_monthly'] 
    df['total_recharge_freq_per_usr'] = df['total_rech_num_monthly'] 


    df = df.drop(['total_rech_amt_monthly','total_rech_num_monthly','total_rech_num_6',
     'total_rech_num_7',
     'total_rech_num_8',
     'total_rech_amt_6',
     'total_rech_amt_7',
     'total_rech_amt_8','amt_per_recharge_6', 'amt_per_recharge_7', 'amt_per_recharge_8'],axis=1)

    date_rech = df.columns[df.columns.str.contains(r"date_of_last_rech_\d{1}")]
    date_rech

    for i in date_rech :
        df[i] = pd.to_datetime(df[i],format="%m/%d/%Y")

    recharge_duration=['prev_recharge_duration','next_recharge_duration']
    df[recharge_duration] = df[date_rech].diff(axis=1).iloc[:,1:]

    df = df.drop(columns = date_rech)

    df[recharge_duration] = df[recharge_duration].apply(lambda x : x.dt.days)

    df['prev_recharge_duration'] = df['prev_recharge_duration'].fillna(0)
    df['next_recharge_duration'] = df['next_recharge_duration'].fillna(0)

    max_rech_cols = [i for i in recharge_cols if re.search(r"max_rech",i)]
    print(max_rech_cols)
    df = df.drop(columns=max_rech_cols)

    df = df.drop(['last_day_rch_amt_6',
           'last_day_rch_amt_7', 'last_day_rch_amt_8'],axis=1)

    date_data_rech = df.columns[df.columns.str.contains(r"date_of_last_rech_data_\d{1}")]
    date_data_rech 

    df = df.drop(date_data_rech,axis=1)

    rech_data =  [i for i in recharge_cols if re.search(r"total_rech_data",i)]
    rech_data

    count_rech_data = [i for i in df.columns if re.search(r'^count_rech_[2-3]g',i)]
    count_rech_data

    avg_rech_data = [i for i in df.columns if re.search(r'^av_rech_amt',i)]
    avg_rech_data

    df[avg_rech_data] = df[avg_rech_data].fillna(0)
    df[rech_data] = df[rech_data].fillna(0)

    df = df.drop(columns=['count_rech_2g_6', 'count_rech_2g_7',
           'count_rech_2g_8', 'count_rech_3g_6', 'count_rech_3g_7',
           'count_rech_3g_8'])

    df['data_per_recharge_monthly'] = (df[avg_rech_data].fillna(0).sum(axis=1) / df[rech_data].fillna(0).sum(axis=1)).fillna(0)

    for amt,freq in zip(avg_rech_data,rech_data) : 
         df[f'data_per_recharge_'+ amt.split('_')[-1]] = df[amt] / df[freq]

    data_per_recharge  = [i for i in df.columns if re.search(r"data_per_recharge_[6-8]?",i)]
    data_per_recharge

    df['data_per_recharge_diff_7_6'] = df['data_per_recharge_7'].fillna(0) - df['data_per_recharge_6'].fillna(0)
    df['data_per_recharge_diff_8_7'] = df['data_per_recharge_8'].fillna(0) - df['data_per_recharge_7'].fillna(0)

    df['data_per_recharge_phase_drift'] = df['data_per_recharge_8'].fillna(0) - ((df['data_per_recharge_7'].fillna(0) + df['data_per_recharge_6'].fillna(0))/2)

    data_drop = avg_rech_data+rech_data
    data_drop

    df = df.drop(columns = data_drop)

    df = df.drop(columns= ['og_others_6', 'og_others_7', 'og_others_8', 'ic_others_6',
           'ic_others_7', 'ic_others_8'])

    df = df.drop( ['data_per_recharge_6',
     'data_per_recharge_7',
     'data_per_recharge_8'],axis=1)

    vol_cols = [i for i in df.columns if re.search(r"vol_",i)]
    vol_cols

    order_list = []
    for j in [6,7,8] : 
        order_list.append([i for i in vol_cols if re.search(r'vol_[2-3]g_mb_{}'.format(j),i)])
    order_list

    for i,j in zip(order_list,[6,7,8]) : 
        df['vol_mb_{}'.format(j)] = df[i].sum(axis=1)

    df = df.drop(vol_cols,axis=1)

    vol_cols = [i for i in df.columns if re.search(r"vol_",i)]
    vol_cols

    df['vol_mb_diff_7_6'] = df['vol_mb_7'] - df['vol_mb_6']
    df['vol_mb_diff_8_7'] = df['vol_mb_8'] - df['vol_mb_7']


    df['vol_mb_phase_drift'] = df['vol_mb_8']  - ((df['vol_mb_7'] + df['vol_mb_6'])/2)

    df['vol_mb_monthly'] = df[vol_cols].sum(axis=1) / len(vol_cols)

    df['mobile_data_usg_per_usr'] = df[vol_cols].sum(axis=1)

    df = df.drop(vol_cols,axis=1)

    ### Monthly Schemes

    monthly_schemes = df.columns[df.columns.str.contains(r"^monthly")]
    {i:df[i].unique() for i in monthly_schemes}

    monthly_6 = [i for i in monthly_schemes if re.search(r"_6$",i)]
    monthly_7 = [i for i in monthly_schemes if re.search(r"_7$",i)]
    monthly_8 = [i for i in monthly_schemes if re.search(r"_8$",i)]

    df['monthly_6'] = df[monthly_6].sum(axis=1)
    df['monthly_7'] = df[monthly_7].sum(axis=1)
    df['monthly_8'] = df[monthly_8].sum(axis=1)

    df = df.drop(monthly_6 + monthly_7 + monthly_8,axis=1)

    monthly_schemes = df.columns[df.columns.str.contains(r"^monthly")]
    monthly_schemes

    df['total_monthly_schemes_per_usr'] = df[monthly_schemes].sum(axis=1) 

    ### Sachet Schemes

    sachet_schemes = df.columns[df.columns.str.contains(r"^sachet")]
    {i:df[i].unique() for i in sachet_schemes}

    sachet_6 = [i for i in sachet_schemes if re.search(r"_6$",i)]
    sachet_7 = [i for i in sachet_schemes if re.search(r"_7$",i)]
    sachet_8 = [i for i in sachet_schemes if re.search(r"_8$",i)]

    df['sachet_6'] = df[sachet_6].sum(axis=1)
    df['sachet_7'] = df[sachet_7].sum(axis=1)
    df['sachet_8'] = df[sachet_8].sum(axis=1)

    df = df.drop(sachet_6 + sachet_7 + sachet_8,axis=1)
    sachet_schemes = df.columns[df.columns.str.contains(r"^sachet")]
    sachet_schemes

    df['total_sachet_schemes_per_usr'] = df[sachet_schemes].sum(axis=1) 

    binary_schemes = df.columns[df.columns.str.contains(r"^(fb|night)")]
    {i:df[i].unique() for i in binary_schemes}

    df['missing_indicator_mobile_data_jun'] = np.where(df['night_pck_user_6'].isnull(),1,0)
    df['missing_indicator_mobile_data_jul'] = np.where(df['night_pck_user_7'].isnull(),1,0)
    df['missing_indicator_mobile_data_aug'] = np.where(df['night_pck_user_8'].isnull(),1,0)

    df[binary_schemes] = df[binary_schemes].fillna(0)

    vbc_monthly = ['aug_vbc_3g',
           'jul_vbc_3g', 'jun_vbc_3g']
    df['total_vbc_per_usr'] = df[vbc_monthly].sum(axis=1)
    df['vbc_monthly'] = df['total_vbc_per_usr'] / len(vbc_monthly)

    df['vbc_diff_7_6'] = df['jul_vbc_3g'] - df['jun_vbc_3g']
    df['vbc_diff_8_7'] = df['aug_vbc_3g'] - df['jul_vbc_3g']

    df = df.drop(columns=['aug_vbc_3g',
           'jul_vbc_3g', 'jun_vbc_3g'])

    monthly_cols = [i for i in df.columns if re.search(r'monthly$',i)]
    diff_cols = [i for i in df.columns if re.search(r'diff',i)]
    per_usr = [i for i in df.columns if re.search(r'_per_usr$',i)]
    phase_cols = [i for i in df.columns if re.search(r'_phase_drift$',i)]

    per_usr = [i for i in per_usr if i not in ['total_monthly_schemes_per_usr',
     'total_sachet_schemes_per_usr']]
    per_usr

    cat_cols = ['monthly_6', 'monthly_7', 'monthly_8',
           'total_monthly_schemes_per_usr', 'sachet_6', 'sachet_7', 'sachet_8',
           'total_sachet_schemes_per_usr']

    per_usr = [i for i in df.columns if re.search(r'_per_usr$',i)]
    per_usr

    for i in per_usr : 
        ext = re.sub(r'(total_|_per_usr)','',i)
        df[ext + '_rate'] = df[i] / df['aon']



    df = df.drop(per_usr,axis=1)

    rate_cols = [i for i in df.columns if re.search(r'_rate$',i)]
    rate_cols

    df['total_mins_incoming_outgoing'] = df['incoming_mins_usg_rate'] + df['incoming_mins_usg_rate']

    # It seems the rate cols are not of much use drop them 
    df = df.drop(columns = rate_cols,axis=1)

    cat_cols = [i for i in cat_cols if not i in ['total_monthly_schemes_per_usr','total_sachet_schemes_per_usr'] ]

    df['high_value_customer']= np.where(df['total_arpu_monthly'].between(arpu_l,arpu_r) & df['amt_per_recharge_monthly'].between(rech_l,rech_r),1,0)

    df = df.drop(columns = 'arpu_avg_7_6')
    
    return df.reset_index(drop=True)

In [ ]:
df_test = test_data()
high_value_customer2 = df_test.pop('high_value_customer')
idc2 = df_test.pop('id')
idc2 = idc2.reset_index(drop=True)
high_value_customer2 = high_value_customer2.reset_index(drop=True)

missing_indicators = [i for i in df_test.columns if re.search(r'^(missing_indicator|night|fb|total_mins)',i)]
numeric_cols = np.setdiff1d(df_test.columns,missing_indicators)
df_test = df_test.reset_index(drop=True)
df_test_cat = df_test[missing_indicators]
df_test = df_test[numeric_cols]
df_test2 = scaler.transform(df_test)
df_test = pd.DataFrame(df_test2,columns=df_test.columns)
df_test = pd.concat([df_test,df_test_cat],axis=1)
# pca_test2 = pd.DataFrame(pca.transform(df_test[numeric_cols]))
# pca_test2.columns = ["pc_" + str(i) for i in range(1,43)]
# pca_test2 = pd.concat([pca_test2 , df_test[missing_indicators]],axis=1)

### Kaggle Final Submission - Accuracy Based (Solely focuses on accuracy) , not the overall best model
- Kaggle Submission is based on accuracy
- Since that is the only metric evaluated there
- This model solely focuses on accuracy while ignoring recall 

In [ ]:

xgb = XGBClassifier(criterion = 'binary:logistic' , eta = 1,
  gamma = 0.1,
  reg_lambda = 10,
  scale_pos_weights = 1 )
xgb,probability = Model_Ensemble(xgb,X_train,y_train,X_test)
# X_train = pd.concat([X_train,X_test])
# y_train = pd.concat([y_train,y_test])

In [ ]:
proba = xgb.predict_proba(df_test)[:,1]
y_pred = pd.Series(proba).apply(lambda x : 1 if x>=0.7 else 0)

In [ ]:
final_test = pd.concat([idc2,y_pred],axis=1)

In [ ]:
final_test.columns = ['id','churn_probability']

In [ ]:
final_test.to_csv('submission_kaggle.csv',index=False)

### Upgrad Final Submission - Overall Best Model (Higher Metrics across the board)
- The Objective here is to predict the high value customers who churn , the recall in this model is the highest so far 
- This model makes sure that all the metrics involved are balanced
- At threshold .36 we can see the best balance when recall is high 

In [ ]:
rf = RandomForestClassifier(criterion = 'gini',
 max_depth= 80,
 min_samples_leaf =  30,
 min_samples_split = 80,
 n_estimators= 50)
rf,probability = Model_Ensemble(rf,X_train,y_train,X_test)

### Test Data Submission (unseen)

In [ ]:
proba = rf.predict_proba(df_test)[:,1]
y_pred = pd.Series(proba).apply(lambda x : 1 if x>=0.36 else 0)


In [ ]:
final_test = pd.concat([idc2,y_pred],axis=1)

In [ ]:
final_test.columns = ['id','churn_probability']

In [ ]:
final_test.to_csv('data/submission.csv', index=False)


### This is unseen data too but was taken from the train test split of the training data
- The metrics are present above too , also printed here just for better view

In [ ]:
tn,fn,fp,tp,accuracy,precision,specificity,senstivity,fpr,y_pred = classification_metrics(y_test,probability,.36)

In [ ]:
accuracy_list,specificity_list,senstivity_list,precision_list = threshold_lists(y_test,probability)
roc_auc_plot(senstivity_list,specificity_list)

### Feature Importance of Upgrad Submission

In [ ]:
fimp = pd.Series(dict(zip(X_train.columns,rf.feature_importances_))).sort_values(ascending=False)

In [ ]:
fimp = fimp[fimp.cumsum().diff()[(fimp.cumsum().diff() > .01) |(fimp.cumsum().diff().isnull()) ].index]

In [ ]:
fimp

In [ ]:
plt.figure(figsize=(15,5))
sns.pointplot(x=fimp.index,y=fimp,color='r')
plt.xticks(rotation=90)
plt.axhline(0.04,color='b')
plt.show()

### Important Variables for prediction 
- loc_mou_monthly
- loc_mou_diff_8_7
- loc_mou_phase_drift
- amnt_per_recharge_phase_drift
- total_mou_phase_drift
- std_mou_diff_8_7
- arpu_phase_drift
#### During EDA it was observed that loc_mou , arpu_phase_drift , amt_per_recharge_phase_drift had high discriminatory powers and it also makes sense intuitively

### Suggestions to Minimize churn
- Sudden drop in minutes of usage (mou) , amount per recharge or arpu of the customer might mean that they have a strong chance  to churn 
- It might be that the price is high or don't they like the current packs or schemes
- Customers that drastically decrease their mou or recharge amount between the months is a clear indication that they are dissatisfied with the service , so sending feedback forms and improving in areas they are dissatisfied with might be a good practice 
- Pay special attention to the customers identified by the Feature high_value_customer(Derived Feature) in the code . Using the high value customer (1 if customer is high value) the churn customers can be prioritized and specialized measures can be taken regarding these customers